1. One Hot Encoding, sector, balcony, agePossession, furnishing type, luxury category and floor category features because we will train a Linear Regression model as our baseline model.
2. we will standardize the numerical features.
3. Log Transformation of price (target) column.

In [39]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR # Support Vector Regression
import pandas as pd
import numpy as np

In [40]:
df = pd.read_csv('A:/CODES/PROJECTS/appartments/data/external/gurgaon/properties_post_feature_selection.csv')
df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,price
0,0.0,113.0,2.0,2.0,2.0,3.0,1105.0,0.0,0.0,1.0,2.0,0.0,0.9
1,0.0,85.0,4.0,4.0,4.0,0.0,2600.0,1.0,0.0,2.0,0.0,2.0,3.2
2,1.0,20.0,9.0,9.0,4.0,2.0,9000.0,0.0,0.0,0.0,1.0,2.0,6.0
3,0.0,105.0,3.0,4.0,3.0,3.0,1765.0,1.0,0.0,0.0,0.0,2.0,0.9
4,0.0,57.0,4.0,5.0,3.0,0.0,3356.0,1.0,0.0,1.0,1.0,2.0,3.6


In [41]:
X = df.drop(columns=['price'])
y = df['price']

In [42]:
columns_to_encode = ['sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [43]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [44]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['property_type', 'bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [45]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', SVR(kernel='rbf'))
    # ('regressor', LinearRegression())
])

In [46]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

scores.mean()

c:\Users\Lenovo\miniconda3\envs\apart\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Lenovo\miniconda3\envs\apart\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Lenovo\miniconda3\envs\apart\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


np.float64(0.8846148606966114)

In [47]:
# Here, we got an R2 score of 88% on baseline model with low standard deviation of 0.01, thus results are consistent
scores.std()

np.float64(0.01760764533429481)

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

In [ ]:
y_pred = np.expm1(y_pred) # to reverse the log transformation

In [50]:
mean_absolute_error(np.expm1(y_test), y_pred)

0.5654018216968131

Low value of MAE indicates that our model giving error of 56 lakhs or 0.56 crores on average and we need to reduce it